Install Libraries


#Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn

# Create Config File

In [ ]:
yaml_config =  """
               model_path: "openai/clip-vit-large-patch14"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities
print(probs)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tensor([[9.9924e-01, 7.6021e-04]], grad_fn=<SoftmaxBackward0>)


# Build Model

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel
from omegaconf import OmegaConf

class ZeroShotClassifier:
  """
  Thực hiện Zero - shot CLIP
  """
  labels = [
      "a photo of cat",
      "a photo of dog",
      "a photo of chicken",
  ]
  def __init__(self, config_path : str) -> None:
    """
    Khởi tạo mô hình Zero-shot
    Args:
      config_path: Đường dẫn đến file cấu hình

    Returns:
      None
    """
    self.config = OmegaConf.load(config_path)
    self.model = CLIPModel.from_pretrained(self.config.model_path)
    self.processor = CLIPProcessor.from_pretrained(self.config.model_path)

  def __call__(self, imgage : Image.Image, k : int) -> list:
    """
    Thực hiện Zero-shot trên một ảnh
    Args:
      image : Ảnh cần dự đoán
      k : top k nhãn chính xác nhất

    Returns:
      list: Danh sách các top k dictionary chứa nhãn và độ chính xác
    """

    inputs = self.processor(text = self.labels,images = image, return_tensors = "pt", padding = True)
    outputs = self.model(**inputs)

    logits_per_image = outputs.logits_per_image # this is the image-text similarity score
    probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities

    top_probs, top_labels = probs.topk(k, dim=1)
    # Ép kiểu tensor về list
    top_probs_list = top_probs.detach().cpu().tolist()[0]
    top_labels_list = top_labels.detach().cpu().tolist()[0]

    results = []
    for prob, index in zip(top_probs_list, top_labels_list):
      label_name = self.labels[index]
      results.append({
            "label": label_name,

            "confidence": round(prob * 100, 2)
        })
    return results


# Initialize Model

In [ ]:
zero_shot = ZeroShotClassifier("./config.yaml")

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
url = "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg"
res = zero_shot(url, 3)
print(res)

[{'label': 'a photo of cat', 'confidence': 99.06}, {'label': 'a photo of chicken', 'confidence': 0.78}, {'label': 'a photo of dog', 'confidence': 0.17}]


# Initialize API

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import requests
import nest_asyncio
import uvicorn
import threading
import asyncio
import time

nest_asyncio.apply()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

def process(path_image: str, k: int):
    """Tải ảnh + chạy inference — toàn bộ trong thread riêng"""
    image = Image.open(requests.get(path_image, stream=True).raw)
    return zero_shot(image, k)

@app.get('/predict')
async def predict(path_image: str, k: int = 3):
    try:
        loop = asyncio.get_event_loop()
        # Đưa cả tải ảnh lẫn inference ra thread riêng
        res = await loop.run_in_executor(None, process, path_image, k)
        return res
    except Exception as e:
        return {"error": str(e)}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print("Server started on http://0.0.0.0:8000")

INFO:     Started server process [5161]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server started on http://0.0.0.0:8000


# Call Local API

In [ ]:
import requests
import threading

result = {}

def call_api():
    API_URL = "http://127.0.0.1:8000/predict"
    params = {
        "path_image": "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg",
        "k": 3
    }
    response = requests.get(API_URL, params=params, timeout=120) 
    result['data'] = response.json()

t = threading.Thread(target=call_api)
t.start()
t.join(timeout=120)

print(result.get('data', 'Timeout hoặc lỗi'))

INFO:     127.0.0.1:41772 - "GET /predict?path_image=https%3A%2F%2Fimages.pexels.com%2Fphotos%2F13663678%2Fpexels-photo-13663678.jpeg&k=3 HTTP/1.1" 200 OK
[{'label': 'a photo of cat', 'confidence': 99.06}, {'label': 'a photo of chicken', 'confidence': 0.78}, {'label': 'a photo of dog', 'confidence': 0.17}]


# Get Pinggy URL

In [ ]:
import subprocess, threading, time, re

public_url = None

def start_pinggy():
    global public_url
    process = subprocess.Popen(
        ["ssh", "-p", "443", "-R", "0:localhost:8000",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=30", "a.pinggy.io"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in process.stdout:
        print(line, end="")
        # bắt .pinggy.link lẫn .pinggy-free.link
        match = re.search(r'https://\S+\.pinggy[\w-]*\.link', line)
        if match:
            public_url = match.group(0)
            print(f"\n Public URL: {public_url}/predict")
            break

threading.Thread(target=start_pinggy, daemon=True).start()
time.sleep(5)
print(f"Dùng URL: {public_url}/predict")

Pseudo-terminal will not be allocated because stdin is not a terminal.
Allocated port 7 for remote forward to localhost:8000
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://ergyf-34-142-148-46.run.pinggy-free.link
https://ergyf-34-142-148-46.run.pinggy-free.link

 Public URL: https://ergyf-34-142-148-46.run.pinggy-free.link/predict
Dùng URL: https://ergyf-34-142-148-46.run.pinggy-free.link/predict


# Call Public API

In [ ]:
import requests
import threading

result = {}

def call_api():
    API_URL = public_url + "/predict"
    params = {
        "path_image": "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg",
        "k": 3
    }
    response = requests.get(API_URL, params=params, timeout=120)
    result['data'] = response.json()

t = threading.Thread(target=call_api)
t.start()
t.join(timeout=120)

print(result.get('data', 'Timeout hoặc lỗi'))

INFO:     34.142.148.46:0 - "GET /predict?path_image=https%3A%2F%2Fimages.pexels.com%2Fphotos%2F13663678%2Fpexels-photo-13663678.jpeg&k=3 HTTP/1.1" 200 OK
[{'label': 'a photo of cat', 'confidence': 99.06}, {'label': 'a photo of chicken', 'confidence': 0.78}, {'label': 'a photo of dog', 'confidence': 0.17}]
